In [1]:
from mpi4py import MPI
from petsc4py import PETSc

import ufl 
from dolfinx import fem, plot
from dolfinx.fem.petsc import LinearProblem
from dolfinx.io import gmshio, XDMFFile 

# load mesh 
rank = 0
filename = "/home/andreasstillits/coding/MscThesis/fempipeline/data/sphere.msh" 
mesh, cell_tags, facet_tags = gmshio.read_from_msh(filename, MPI.COMM_WORLD, rank, gdim=3)

# define function space
V = fem.functionspace(mesh, ("Lagrange", 1))
dx = ufl.Measure("dx", domain=mesh, subdomain_data=cell_tags)
ds = ufl.Measure("ds", domain=mesh, subdomain_data=facet_tags)

VOLUME_TAG = 1 
TOP_SURFACE_TAG = 2
BOTTOM_SURFACE_TAG = 3
CURVED_SURFACE_TAG = 4
MESOPHYLL_SURFACE_TAG = 5

SURFACE_TAGS = [2, 3, 4, 5]

u0 = fem.Function(V) 
u0.x.array[:] = 0.0
u1 = fem.Function(V)
u1.x.array[:] = 1.0
bcs = []

for tag in SURFACE_TAGS:
    facets = facet_tags.find(tag)
    dofs = fem.locate_dofs_topological(V, mesh.topology.dim - 1, facets)
    if tag < 5:
        bcs.append(fem.dirichletbc(u0, dofs))
    else:
        bcs.append(fem.dirichletbc(u1, dofs))

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# bilinear form 
a = ufl.inner(ufl.grad(u), ufl.grad(v)) * dx(VOLUME_TAG)
L = fem.Constant(mesh, PETSc.ScalarType(0.0)) * v * dx(VOLUME_TAG)

problem = LinearProblem(a, L, bcs=bcs, petsc_options={"ksp_type": "cg", "pc_type": "hypre"})
uh = problem.solve()

with XDMFFile(MPI.COMM_WORLD, "solution.xdmf", "w") as xdmf:
    xdmf.write_mesh(mesh)
    xdmf.write_function(uh, 0.0)

# print solution using pyvista 

if rank == 0:
    import pyvista as pv
    topology, cell_types, geometry = plot.vtk_mesh(mesh, mesh.topology.dim)
    grid = pv.UnstructuredGrid(topology, cell_types, geometry)
    grid.point_data["uh"] = uh.x.array.real

    xmin, xmax, ymin, ymax, zmin, zmax = grid.bounds
    slices = grid.slice_orthogonal(x=(xmin+xmax)/2, y=(ymin+ymax)/2, z=(zmin+zmax)/2)
    p = pv.Plotter()
    p.add_mesh(slices, scalars="uh")
    p.add_mesh(grid.outline(), color="k")
    p.show()



Info    : Reading '/home/andreasstillits/coding/MscThesis/fempipeline/data/sphere.msh'...
Info    : 611 entities
Info    : 20385 nodes
Info    : 117000 elements
Info    : Done reading '/home/andreasstillits/coding/MscThesis/fempipeline/data/sphere.msh'                      


Widget(value='<iframe src="http://localhost:43775/index.html?ui=P_0x733f78874830_0&reconnect=auto" class="pyvi…